In [16]:
%reload_ext autoreload
%autoreload 2

%matplotlib inline

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


if Path.cwd().name in {"notebooks", "research"}:
    os.chdir(Path.cwd().parent)

from src.data.data_manager import DataManager
from src.analytics.factor_model import FactorEngine

project_root = str(Path.cwd())
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"CWD: {Path.cwd()}")
print(f"Data path exists: {(Path.cwd() / 'data' / 'raw').exists()}")

CWD: c:\Users\reymo\quant-portfolio
Data path exists: True


In [17]:
from src.analytics.macro_bl import MacroBLModel
from src.data.data_manager import DataManager
from src.analytics.mvo_optimizer import MeanVarianceOptimizer
from src.analytics.black_litterman import BlackLittermanModel

dm = DataManager()
tickers = dm.list_existing_tickers()
returns = dm.load_returns(tickers=tickers, start_date='2020-01-01', end_date='2025-12-31')
market_caps = dm.get_latest_market_cap(tickers=tickers)

bl_helper = BlackLittermanModel(market_caps=market_caps)
mvo = MeanVarianceOptimizer(returns=returns)
sigma = mvo._apply_ledoit_wolf_shrinkage()
sigma_ann = sigma * 252.0
sigma_ann_df = pd.DataFrame(sigma_ann, index=tickers, columns=tickers)

macro_bl = MacroBLModel(returns=returns, covariance=sigma_ann_df, market_caps=market_caps)
Pi = bl_helper.calculate_implied_returns(covariance=sigma_ann_df)

macro_bl.inject_factor_view(
    factor_index=3, 
    relative_return=0.08, 
    confidence=0.6
)

post_mu = macro_bl.compute_posterior()

INFO:FactorEngine:PCA Factors computed. Top PC Variance: 0.007179


In [18]:
comparison_df = post_mu - Pi
comparison_df

AAPL     -0.011023
ABBV     -0.004081
ADBE     -0.006829
AIR.PA    0.008643
AMZN     -0.010062
ASML     -0.008292
AVGO     -0.020191
BABA     -0.001160
BNP.PA    0.002925
CRM      -0.008388
CSCO     -0.011815
CVX      -0.036905
DG.PA     0.008539
DIS      -0.018486
GOOGL    -0.011704
HD       -0.007214
INTC     -0.022043
JNJ      -0.000896
JPM      -0.018361
KO       -0.002234
LIN      -0.004069
MA       -0.010914
MC.PA     0.024540
MCD      -0.005387
META     -0.010685
MRK      -0.000555
MSFT     -0.009707
NFLX     -0.010001
NKE      -0.003646
NVDA     -0.020314
NVO       0.008609
OR.PA     0.024295
ORCL     -0.012480
PEP      -0.001236
PFE      -0.000258
PG        0.001650
RIO      -0.020245
RMS.PA    0.026120
SAN.PA    0.013214
SAP       0.001565
SHEL     -0.038987
SU.PA     0.017824
TM       -0.007046
TSLA     -0.023164
TTE      -0.031724
TTE.PA   -0.016598
UL        0.004182
UNH      -0.004107
V        -0.008917
XOM      -0.038559
dtype: float64

In [14]:
# Delta Mu is the shift in returns
delta_mu = post_mu - Pi

# Project the shift back onto the Factor Eigenvectors
# eigenvectors shape: (n_assets, n_factors)
factor_shifts = delta_mu.values @ macro_bl.eigenvectors

# Create a Series to visualize which "Macro Lever" actually moved
factor_names = [f"PC{i+1}" for i in range(factor_shifts.shape[0])]
shift_verification = pd.Series(factor_shifts, index=factor_names)

print(shift_verification)

PC1   -0.054372
PC2    0.004002
PC3    0.005394
PC4    0.096325
PC5   -0.008038
dtype: float64


In [20]:
factor_limits = {0: 1, 1: 0.2, 2: 0.2, 4: 0.2}
weights = mvo.solve(expected_returns=post_mu, covariance=sigma_ann_df, factor_exposure=macro_bl.eigenvectors, factor_limits=factor_limits)

ValueError: Optimization failed: Positive directional derivative for linesearch